Ensure the basic catalog structure is in place

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS league_pipeline;
CREATE SCHEMA IF NOT EXISTS league_pipeline.landing_zone;
CREATE SCHEMA IF NOT EXISTS league_pipeline.raw;
CREATE SCHEMA IF NOT EXISTS league_pipeline.bronze;
CREATE VOLUME IF NOT EXISTS league_pipeline.landing_zone.players;
CREATE VOLUME IF NOT EXISTS league_pipeline.landing_zone.checkpoints;

Infer Schema before-hand

In [ ]:
import json

SCHEMA_FILE_PATH = "/Volumes/league_pipeline/landing_zone/checkpoints/players/player_schema.json"
LANDING_PATH = "/Volumes/league_pipeline/landing_zone/players/"

files = dbutils.fs.ls(LANDING_PATH)
new_schema_json = None

if len(files) > 0:
    new_schema_json = spark.read.format("parquet").load(LANDING_PATH).schema.json()
    
    with open(SCHEMA_FILE_PATH, "w") as f:
        f.write(new_schema_json)
    print(f"Schema refreshed from {LANDING_PATH}")
else:
    print("No files in landing zone — reusing existing schema.")

if new_schema_json:
    print(new_schema_json)

Auto Loader Job

In [0]:
import json
from pyspark.sql.types import StructType
from pyspark.sql.functions import col, current_timestamp

SCHEMA_FILE_PATH = "/Volumes/league_pipeline/landing_zone/checkpoints/players/player_schema.json"
LANDING_PATH = "dbfs:/Volumes/league_pipeline/landing_zone/players/"
CHECKPOINT_SCHEMA = "dbfs:/Volumes/league_pipeline/landing_zone/checkpoints/players/schema"
CHECKPOINT_DATA = "dbfs:/Volumes/league_pipeline/landing_zone/checkpoints/players/data"
RAW_TABLE = "league_pipeline.raw.players"

with open(SCHEMA_FILE_PATH) as f:
    player_schema = StructType.fromJson(json.load(f))
    
prior_max_ts = (
    spark.table(RAW_TABLE)
    .agg({"_ingested_at": "max"})
    .collect()[0][0]
)
if prior_max_ts is None:
    prior_max_ts = "1900-01-01T00:00:00.000Z"

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", CHECKPOINT_SCHEMA)
    .schema(player_schema)
    .load(LANDING_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

query = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_DATA)
    .trigger(availableNow=True)
    .toTable(RAW_TABLE)
)
query.awaitTermination()

just_ingested = (
    spark.table(RAW_TABLE)
    .filter(col("_ingested_at") >= prior_max_ts) # When the query started
    .select("_source_file")
    .distinct()
    .collect()
)

deleted, failed = [], []
for row in just_ingested:
    path = row["_source_file"]
    try:
        dbutils.fs.rm(path)
        deleted.append(path)
    except Exception as e:
        failed.append((path, str(e)))

print(f"Deleted {len(deleted)} files from landing zone.")
if failed:
    print(f"WARNING: {len(failed)} files failed to delete: {failed}")